# Preliminary Steps

We'll be using the statsmodel api for our data this week. To install the statsmodel package, activate your conda environment in your terminal and run conda install -c conda-forge statsmodels.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

Here is the dataset we'll be working with today and its details: https://www.statsmodels.org/stable/datasets/generated/fair.html

In [ ]:
# load dataset
affairs = sm.datasets.fair.load_pandas().data

# add "affair" column: 1 represents having affairs, 0 represents not
affairs['affair'] = (affairs.affairs > 0).astype(int)

In [ ]:
affairs

The goal of a logistic regression using this data is to see if we can successfully predict whether a woman will have an affair or not (which is why we created a boolean column for an affair).

# EDA

Let's check the difference in averages for those that had affairs and those that didn't. You can do this by grouping by the affairs column we made earlier and using the mean method.

In [ ]:
# TODO

As we can see, those who had affairs tend to rate their marriage lower, spent longer married, etc. Make sure you understand what each value in each column means here!

Now, check the average values of each column grouped by rate_marriage.

In [ ]:
# TODO

You should see three variables that increase as the marriage rating declines. What are they?

**Your answer here**

# Visualizations

Out of curiosity, let's check the education level and frequency of each education level in the data.

In [ ]:
# TODO

Create another histogram for marriage rating!

In [ ]:
# TODO

Let's take a look at a more useful barplot still using marriage rating (organized by whether they had an affair or not). What do we learn from this?

In [ ]:
pd.crosstab(affairs.rate_marriage, affairs.affair.astype(bool)).plot(kind='bar')
plt.title('Marriage Rating Distribution by Affair Status')
plt.xlabel('Marriage Rating')
plt.ylabel('Frequency')

Finally, let's take a look at the percentage of people who had affairs by the number of years they were married.

In [ ]:
affair_yrs_married = pd.crosstab(affairs.yrs_married, affairs.affair.astype(bool))
affair_yrs_married.div(affair_yrs_married.sum(1).astype(float), axis=0).plot(kind='bar', stacked=True)
plt.title('Affair Percentage by Years Married')
plt.xlabel('Years Married')
plt.ylabel('Percentage')

Interpret this graph. What does this mean/tell us?

**Your answer here**

# Logistic Regression Prep

Let's import some necessary packages first. <br>
If you don't have sklearn installed, use conda install scikit-learn in your conda environment

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics

We saw that occupation and occupation_husb are technically categorical variables describing the type of job each person has. We would like to include it as a predictor variable, but it would not make sense to use it in a continuous numerical variable (since each number is a category). Therefore, we must make these columns into dummy variables (if you're confused on any of this, ask us questions!)

In [ ]:
affairs

If we try using the [get_dummies()](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) method on our data, we'll notice that it doesn't work as expected (dimensions of our data are the same!)

In [ ]:
pd.get_dummies(affairs)

This is because the data in the columns we want dummies for is a float. However, pd.get_dummies automatically converts categorical variables and not quantitative variables. To fix this, turn the columns of interest into strings to make it categorical. This is called one hot encoding.

In [ ]:
# Create a new dataframe called ohc_affairs (ohc stands for One Hot Encoding) where we have all the same data as before,
# but we have one hot encoded the categorical columns (read above to see which columns here are categorical)
# Use pd.get_dummies and use the documentation!
# Hint: You should have 32 columns after you do your one hot encoding

# Logistic Regression

Now that we have our variables let's test our model before using a train/test split.

In [ ]:
# TODO
# y = ?
# X = ?

Create an sklearn LogisticRegression object and fit it to your design matrix X and your target feature y.

In [ ]:
# TODO

Find your model score using the score() method of the LogisticRegression object.

In [ ]:
# TODO

This accuracy seems high, but we need to check something called the null error rate. The null error rate is the accuracy we would get by only guessing one type of answer (in this case, if we only guessed that a person would not have an affair).

In [ ]:
# TODO

So 32% of women had affairs, meaning 68% of women did not have affairs. Our null error rate is 68%, meaning we could guess accurately 68% of the time by only predicting that they would not have an affair. Our model is better than this rate only by a slight amount.

Let's check the coefficients of each variable to see what kind of patterns we get.

In [ ]:
pd.DataFrame(zip(X.columns, np.transpose(model.coef_)))

What kind of information can we get from this? I.e. are there certain variables that have a factor in decreasing the likelihood of an affair? Are there variable that increase the likelihood? Think about what it means in this case for a coefficient to be positive or negative.

**Your answer here**

Now time to add in training/testing for a more accurate model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
model2 = LogisticRegression(max_iter=3000)
# TODO
# train your model

In [ ]:
# TODO
# predict class labels for the test set
# predicted = ?


Now let's check the accuracy of our model by comparing the values we predict to the actual ones.

In [ ]:
print(metrics.accuracy_score(y_test, predicted))

Cool, our model is pretty good because even after test/training we see that accuracy is slightly higher but still around 73%.

Now let's get a confusion matrix to see how we did on an individual prediction level. Interpret this below (remember there is a slide on this!). Specifically, what is the TPR and FPR and what do each of the four values mean in the context of this question?

In [ ]:
print(metrics.confusion_matrix(y_test, predicted))

**Your answer here**

Next up is something called a classification report, which basically tells us how accurate our model is. This goes off the confusion matrix with 3 important things: "Precision", which tells us how many people were predicted to have an affair that actually did; "Recall", which tells us the percent of people that actually had an affair and were correctly predicted; and "F1 Score", which combines these two metrics to give us a measure of our model accuracy (closer to 1 is more accurate). "Support" just tells us the count for each variable.

In [ ]:
print(metrics.classification_report(y_test, predicted))

### EXTRA
Run the below code to see what your model is actually outputting behind the scenes. Notice how it doesn't output 1 and 0 like our predict method but instead it outputs probabilities. sklearn's LogisticRegression object automatically classifies something as a 1 if the probability is >= 0.5.
<br>
<br>
If you want to learn more about how logistic regression works, go to the advanced classification model supplemental lecture!


In [ ]:
probs = model2.predict_proba(X_test)[:, 1]
probs

Now try using sklearn's decision tree classifier and compare the score to your logistic regression.

In [ ]:
from sklearn import tree

In [ ]:
clf = tree.DecisionTreeClassifier()
# TODO
# train your decision tree and check the accuracy using the score method